# 🦎 Hyena vs Transformer — Training Walkthrough
**Môn:** CS2308 – Chuyên đề NLP | **Nhóm:** [Điền tên nhóm]

**Notebook này chạy end-to-end trên Google Colab (T4 GPU).**

### Nội dung:
1. Setup môi trường
2. Load và khám phá WikiText-2
3. Train Transformer-small (baseline)
4. Train Hyena-small
5. So sánh kết quả (E1)
6. Đo scaling theo sequence length (E2 + E3)
7. Vẽ biểu đồ tổng hợp

> **Lưu ý:** Chọn Runtime → Change runtime type → **GPU (T4)** trước khi chạy!

## Cell 1: Setup môi trường

In [ ]:
# ── Cài dependencies ──────────────────────────────────────────
!pip install -q torch transformers datasets tqdm

# ── Clone repo (thay bằng URL repo của nhóm) ─────────────────
# !git clone https://github.com/your-username/hyena-reproduction.git
# %cd hyena-reproduction

# ── Hoặc upload thủ công các file .py từ repo ────────────────
# Nếu không dùng git, upload file và chạy từ thư mục đó

import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

## Cell 2: Load và Khám Phá Dataset WikiText-2

In [ ]:
from datasets import load_dataset
from transformers import GPT2Tokenizer
import torch

# ── Load WikiText-2 ──────────────────────────────────────────
print('Loading WikiText-2...')
wt2 = load_dataset('wikitext', 'wikitext-2-raw-v1')
print(wt2)

# ── Xem vài dòng mẫu ────────────────────────────────────────
print('\n--- Sample texts ---')
for i, text in enumerate(wt2['train']['text'][:5]):
    if text.strip():
        print(f'[{i}] {text[:100]}...')

# ── Thống kê ──────────────────────────────────────────────────
print('\n--- Statistics ---')
for split in ['train', 'validation', 'test']:
    n_texts = len(wt2[split])
    total_chars = sum(len(t) for t in wt2[split]['text'])
    print(f'{split:12s}: {n_texts:6,} articles | {total_chars:12,} chars')

In [ ]:
# ── Tokenization ─────────────────────────────────────────────
print('Loading GPT-2 tokenizer...')
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
print(f'Vocab size: {tokenizer.vocab_size:,}')

# Tokenize và đếm tokens
for split in ['train', 'validation', 'test']:
    text = '\n'.join([t for t in wt2[split]['text'] if t.strip()])
    tokens = tokenizer.encode(text)
    print(f'{split:12s}: {len(tokens):>10,} tokens')

print('\n✅ Tokenizer ready!')

## Cell 3: Định nghĩa Data Pipeline

In [ ]:
from torch.utils.data import Dataset, DataLoader

class WikiText2Dataset(Dataset):
    def __init__(self, split='train', seq_len=256, tokenizer=None):
        self.seq_len = seq_len
        if tokenizer is None:
            tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
        # Load và tokenize
        data = load_dataset('wikitext', 'wikitext-2-raw-v1', split=split)
        text = '\n'.join([t for t in data['text'] if t.strip()])
        token_ids = tokenizer.encode(text)
        self.tokens = torch.tensor(token_ids, dtype=torch.long)
        self.n_samples = (len(self.tokens) - 1) // seq_len
        print(f'  [{split}] {len(self.tokens):,} tokens → {self.n_samples:,} samples (seq_len={seq_len})')

    def __len__(self): return self.n_samples

    def __getitem__(self, idx):
        start = idx * self.seq_len
        x = self.tokens[start : start + self.seq_len]
        y = self.tokens[start + 1 : start + self.seq_len + 1]
        return x, y

def make_loaders(seq_len=256, batch_size=16):
    tok = GPT2Tokenizer.from_pretrained('gpt2')
    print(f'Creating dataloaders (seq_len={seq_len}, batch_size={batch_size})...')
    train_ds = WikiText2Dataset('train', seq_len, tok)
    val_ds   = WikiText2Dataset('validation', seq_len, tok)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    print(f'  Train batches: {len(train_loader):,} | Val batches: {len(val_loader):,}')
    return train_loader, val_loader

# Test
SEQ_LEN = 256
BATCH_SIZE = 16
train_loader, val_loader = make_loaders(SEQ_LEN, BATCH_SIZE)
x, y = next(iter(train_loader))
print(f'\nBatch shapes: x={x.shape}, y={y.shape}  ✅')

## Cell 4: Transformer-small Model

In [ ]:
import math
import torch.nn as nn
import torch.nn.functional as F

class CausalSelfAttention(nn.Module):
    """Multi-Head Causal Self-Attention — O(L²) complexity"""
    def __init__(self, d_model, n_heads, max_seq_len, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = math.sqrt(self.d_head)
        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer('causal_mask', mask)

    def forward(self, x):
        B, L, D = x.shape
        q, k, v = self.qkv_proj(x).split(D, dim=-1)
        def sh(t): return t.view(B, L, self.n_heads, self.d_head).transpose(1,2)
        q, k, v = sh(q), sh(k), sh(v)
        attn = (q @ k.transpose(-2,-1)) / self.scale
        attn = attn.masked_fill(~self.causal_mask[:L,:L], float('-inf'))
        attn = self.dropout(F.softmax(attn, dim=-1))
        out = (attn @ v).transpose(1,2).contiguous().view(B,L,D)
        return self.out_proj(out)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, max_seq_len, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, max_seq_len, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model,d_ff), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(d_ff,d_model), nn.Dropout(dropout))
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

class TransformerLM(nn.Module):
    """GPT-like Causal LM | Complexity: O(N_layers · L² · d)"""
    def __init__(self, vocab_size=50257, d_model=256, n_layers=4,
                 n_heads=4, d_ff=1024, max_seq_len=1024, dropout=0.1):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([TransformerBlock(d_model,n_heads,d_ff,max_seq_len,dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight
        self.apply(self._init)
    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)): nn.init.normal_(m.weight, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None: nn.init.zeros_(m.bias)
    def forward(self, x):
        B, L = x.shape
        h = self.drop(self.token_emb(x) + self.pos_emb(torch.arange(L, device=x.device)))
        for blk in self.blocks: h = blk(h)
        return self.lm_head(self.ln_f(h))
    def count_params(self): return sum(p.numel() for p in self.parameters() if p.requires_grad)

# Test
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tf_model = TransformerLM().to(device)
print(f'TransformerLM: {tf_model.count_params():,} parameters')
test_x = torch.randint(0, 50257, (2, SEQ_LEN), device=device)
out = tf_model(test_x)
print(f'Output shape: {out.shape}  ✅')

## Cell 5: Hyena-small Model

In [ ]:
class HyenaFilter(nn.Module):
    """Implicit long filter: h_t = Window(t) · FFN(PosEnc(t))"""
    def __init__(self, d_model, order=2, filter_dim=64):
        super().__init__()
        self.order, self.d_model = order, d_model
        self.ffn = nn.Sequential(nn.Linear(filter_dim, filter_dim), nn.SiLU(),
                                  nn.Linear(filter_dim, order * d_model))
        self.log_decay = nn.Parameter(torch.randn(order, d_model) * 0.1)
        self.bias = nn.Parameter(torch.zeros(order, d_model))
        self.filter_dim = filter_dim

    def _pos_enc(self, L, device):
        t = torch.linspace(0, 1, L, device=device).unsqueeze(-1)
        freqs = torch.exp(torch.linspace(0, math.log(1e4), self.filter_dim//2, device=device))
        return torch.cat([torch.sin(t*freqs), torch.cos(t*freqs)], dim=-1)

    def forward(self, L):
        device = self.log_decay.device
        h = self.ffn(self._pos_enc(L, device)).view(L, self.order, self.d_model).permute(1,2,0)
        t = torch.linspace(0, 1, L, device=device)
        decay = torch.exp(-torch.exp(self.log_decay)).unsqueeze(-1)
        w = decay ** t.unsqueeze(0).unsqueeze(0)
        return h * w + self.bias.unsqueeze(-1) * (1 - w)

class HyenaOperator(nn.Module):
    """Hyena recurrence — O(N·L·log L) complexity"""
    def __init__(self, d_model, order=2, filter_dim=64, max_seq_len=1024, dropout=0.1):
        super().__init__()
        self.order = order
        self.input_proj = nn.Linear(d_model, (order+1)*d_model, bias=False)
        self.short_conv = nn.Conv1d((order+1)*d_model, (order+1)*d_model,
                                     kernel_size=3, padding=2, groups=(order+1)*d_model)
        self.filt = HyenaFilter(d_model, order, filter_dim)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.drop = nn.Dropout(dropout)

    def _fft_conv(self, h, v):
        L = v.shape[-1]; n = 2*L
        return torch.fft.irfft(torch.fft.rfft(h,n=n)*torch.fft.rfft(v,n=n), n=n)[...,:L]

    def forward(self, u):
        B, L, D = u.shape
        p = self.input_proj(u).transpose(1,2)  # (B,(N+1)D,L)
        p = self.short_conv(p)[...,:L].transpose(1,2)  # (B,L,(N+1)D)
        parts = p.chunk(self.order+1, dim=-1)
        gates = [g.transpose(1,2) for g in parts[:-1]]
        z = parts[-1].transpose(1,2)
        h_all = self.filt(L)
        for n in range(self.order):
            z = gates[n] * self._fft_conv(h_all[n], z)
        return self.drop(self.out_proj(z.transpose(1,2)))

class HyenaBlock(nn.Module):
    def __init__(self, d_model, order=2, filter_dim=64, d_ff=1024, max_seq_len=1024, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.hyena = HyenaOperator(d_model, order, filter_dim, max_seq_len, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model,d_ff), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(d_ff,d_model), nn.Dropout(dropout))
    def forward(self, x):
        x = x + self.hyena(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

class HyenaLM(nn.Module):
    """Hyena Language Model | Complexity: O(N_layers · N_order · L log L · d)"""
    def __init__(self, vocab_size=50257, d_model=256, n_layers=4, order=2,
                 filter_dim=64, d_ff=1024, max_seq_len=1024, dropout=0.1):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([HyenaBlock(d_model,order,filter_dim,d_ff,max_seq_len,dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight
        self.apply(self._init)
    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)): nn.init.normal_(m.weight, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None: nn.init.zeros_(m.bias)
    def forward(self, x):
        B, L = x.shape
        h = self.drop(self.token_emb(x) + self.pos_emb(torch.arange(L, device=x.device)))
        for blk in self.blocks: h = blk(h)
        return self.lm_head(self.ln_f(h))
    def count_params(self): return sum(p.numel() for p in self.parameters() if p.requires_grad)

# Test
hy_model = HyenaLM().to(device)
print(f'HyenaLM: {hy_model.count_params():,} parameters')
out = hy_model(test_x)
print(f'Output shape: {out.shape}  ✅')

## Cell 6: Training Loop

In [ ]:
from torch.optim import AdamW
import time, csv, math

def train_model(model, train_loader, val_loader, epochs=15, lr=3e-4,
                model_name='model', device=device):
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.1)
    best_val_loss = float('inf')
    history = []

    for epoch in range(1, epochs+1):
        model.train()
        total_loss, n_batches = 0.0, 0
        t0 = time.time()
        if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1

        train_loss = total_loss / n_batches
        val_loss = eval_loss(model, val_loader, device)
        val_ppl = math.exp(min(val_loss, 20))
        epoch_time = time.time() - t0
        peak_mem = torch.cuda.max_memory_allocated()/1024**2 if torch.cuda.is_available() else 0

        history.append({'epoch': epoch, 'train_loss': train_loss,
                        'val_loss': val_loss, 'val_ppl': val_ppl,
                        'time_s': epoch_time, 'mem_mb': peak_mem})

        print(f'[{model_name}] Epoch {epoch:2d}/{epochs} | '
              f'Train: {train_loss:.4f} | Val: {val_loss:.4f} | '
              f'PPL: {val_ppl:.2f} | Time: {epoch_time:.1f}s | Mem: {peak_mem:.0f}MB')

    return history

def eval_loss(model, loader, device):
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            total += F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1)).item()
            n += 1
    model.train()
    return total / max(1, n)

print('Training functions ready ✅')

## Cell 7: Chạy E1 — Transformer vs Hyena (L=256)

In [ ]:
# ── Cấu hình thực nghiệm E1 ───────────────────────────────────
EPOCHS = 15          # Tăng lên 20-30 nếu có đủ thời gian
SEQ_LEN = 256
BATCH_SIZE = 16      # Giảm xuống 8 nếu OOM
LR = 3e-4

print('='*60)
print(f'E1: Baseline Comparison | L={SEQ_LEN} | Epochs={EPOCHS}')
print('='*60)

# Load dataloaders
train_loader, val_loader = make_loaders(SEQ_LEN, BATCH_SIZE)

# ── Train Transformer ────────────────────────────────────────
print('\n--- Training Transformer-small ---')
tf_model = TransformerLM(max_seq_len=SEQ_LEN).to(device)
tf_history = train_model(tf_model, train_loader, val_loader,
                          epochs=EPOCHS, lr=LR, model_name='Transformer')

# ── Train Hyena ──────────────────────────────────────────────
print('\n--- Training Hyena-small ---')
hy_model = HyenaLM(max_seq_len=SEQ_LEN).to(device)
hy_history = train_model(hy_model, train_loader, val_loader,
                          epochs=EPOCHS, lr=LR, model_name='Hyena')

print('\n✅ E1 Training complete!')

## Cell 8: Tổng Hợp Kết Quả E1

In [ ]:
import pandas as pd

# Tạo bảng kết quả E1
tf_final = tf_history[-1]
hy_final = hy_history[-1]

print('\n' + '='*65)
print('E1 RESULTS: Transformer vs Hyena (WikiText-2, L=256)')
print('='*65)
print(f'{"Metric":<25} {"Transformer":>18} {"Hyena":>18}')
print('-'*65)
print(f'{"Final Train Loss":<25} {tf_final["train_loss"]:>18.4f} {hy_final["train_loss"]:>18.4f}')
print(f'{"Final Val Loss":<25} {tf_final["val_loss"]:>18.4f} {hy_final["val_loss"]:>18.4f}')
print(f'{"Final Val Perplexity":<25} {tf_final["val_ppl"]:>18.2f} {hy_final["val_ppl"]:>18.2f}')
print(f'{"Avg Epoch Time (s)":<25} {sum(h["time_s"] for h in tf_history)/len(tf_history):>18.1f} {sum(h["time_s"] for h in hy_history)/len(hy_history):>18.1f}')
print('='*65)

# Save CSV
import os; os.makedirs('results', exist_ok=True)
rows = []
for h in tf_history:
    rows.append({'model': 'transformer', **h})
for h in hy_history:
    rows.append({'model': 'hyena', **h})
pd.DataFrame(rows).to_csv('results/E1_baseline.csv', index=False)
print('\n💾 Saved: results/E1_baseline.csv')

## Cell 9: Vẽ Biểu Đồ Loss Curves

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.style as style
style.use('seaborn-v0_8-whitegrid')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
epochs_list = [h['epoch'] for h in tf_history]

# ── Plot 1: Training Loss ────────────────────────────────────
axes[0].plot(epochs_list, [h['train_loss'] for h in tf_history], 'b-o', label='Transformer', markersize=4)
axes[0].plot(epochs_list, [h['train_loss'] for h in hy_history], 'r-s', label='Hyena', markersize=4)
axes[0].set_title('Training Loss', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# ── Plot 2: Validation Loss ──────────────────────────────────
axes[1].plot(epochs_list, [h['val_loss'] for h in tf_history], 'b-o', label='Transformer', markersize=4)
axes[1].plot(epochs_list, [h['val_loss'] for h in hy_history], 'r-s', label='Hyena', markersize=4)
axes[1].set_title('Validation Loss', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

# ── Plot 3: Perplexity ───────────────────────────────────────
axes[2].plot(epochs_list, [h['val_ppl'] for h in tf_history], 'b-o', label='Transformer', markersize=4)
axes[2].plot(epochs_list, [h['val_ppl'] for h in hy_history], 'r-s', label='Hyena', markersize=4)
axes[2].set_title('Validation Perplexity', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Perplexity (lower = better)')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.suptitle(f'E1: Transformer vs Hyena | WikiText-2, L={SEQ_LEN}',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
os.makedirs('results/plots', exist_ok=True)
plt.savefig('results/plots/loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: results/plots/loss_curves.png')

## Cell 10: Đo Scaling — E2 (Transformer) và E3 (Hyena)

In [ ]:
import time

def measure_speed(model_class, model_kwargs, seq_len, batch_size=4,
                   n_warmup=3, n_runs=10, device=device):
    """Đo thời gian forward pass và GPU memory."""
    try:
        model = model_class(**{**model_kwargs, 'max_seq_len': seq_len}).to(device)
        model.eval()
        x = torch.randint(0, 50257, (batch_size, seq_len), device=device)

        # Warmup
        with torch.no_grad():
            for _ in range(n_warmup): _ = model(x)

        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()

        # Measure
        times = []
        with torch.no_grad():
            for _ in range(n_runs):
                t0 = time.perf_counter()
                _ = model(x)
                if torch.cuda.is_available(): torch.cuda.synchronize()
                times.append((time.perf_counter() - t0) * 1000)

        avg_ms = sum(times) / len(times)
        mem_mb = torch.cuda.max_memory_allocated()/1024**2 if torch.cuda.is_available() else 0
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        return {'time_ms': avg_ms, 'mem_mb': mem_mb}
    except RuntimeError as e:
        if 'memory' in str(e).lower():
            if torch.cuda.is_available(): torch.cuda.empty_cache()
            return {'time_ms': None, 'mem_mb': None}  # OOM
        raise

# ── Cấu hình model giống nhau ────────────────────────────────
BASE_KWARGS = dict(vocab_size=50257, d_model=256, n_layers=4, d_ff=1024, dropout=0.0)
TF_KWARGS   = {**BASE_KWARGS, 'n_heads': 4}
HY_KWARGS   = {**BASE_KWARGS, 'order': 2, 'filter_dim': 64}

SEQ_LENS = [128, 256, 512, 1024, 2048]
SCALING_BATCH = 4  # Nhỏ để không OOM

print('Measuring scaling... (có thể mất vài phút)\n')
print(f'{"Seq Len":>8} | {"TF Time(ms)":>12} | {"TF Mem(MB)":>12} | {"HY Time(ms)":>12} | {"HY Mem(MB)":>12} | {"Speedup":>8}')
print('-'*80)

scale_results = []
for sl in SEQ_LENS:
    tf_m = measure_speed(TransformerLM, TF_KWARGS, sl, SCALING_BATCH)
    hy_m = measure_speed(HyenaLM, HY_KWARGS, sl, SCALING_BATCH)

    tf_t = f'{tf_m["time_ms"]:.2f}' if tf_m['time_ms'] else 'OOM'
    hy_t = f'{hy_m["time_ms"]:.2f}' if hy_m['time_ms'] else 'OOM'
    tf_mem = f'{tf_m["mem_mb"]:.1f}' if tf_m['mem_mb'] else 'OOM'
    hy_mem = f'{hy_m["mem_mb"]:.1f}' if hy_m['mem_mb'] else 'OOM'
    speedup = f'{tf_m["time_ms"]/hy_m["time_ms"]:.2f}x' if (tf_m['time_ms'] and hy_m['time_ms']) else 'N/A'
    print(f'{sl:>8} | {tf_t:>12} | {tf_mem:>12} | {hy_t:>12} | {hy_mem:>12} | {speedup:>8}')
    scale_results.append({'seq_len': sl, 'tf_time_ms': tf_m['time_ms'], 'tf_mem_mb': tf_m['mem_mb'],
                           'hy_time_ms': hy_m['time_ms'], 'hy_mem_mb': hy_m['mem_mb']})

pd.DataFrame(scale_results).to_csv('results/scaling_comparison.csv', index=False)
print('\n💾 Saved: results/scaling_comparison.csv')

## Cell 11: Biểu Đồ Scaling

In [ ]:
import pandas as pd
df = pd.DataFrame(scale_results)

# Lọc bỏ OOM
df_valid = df.dropna()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Plot 1: Time vs Seq Length ────────────────────────────────
if len(df_valid) > 0:
    axes[0].plot(df_valid['seq_len'], df_valid['tf_time_ms'], 'b-o', label='Transformer O(L²)', linewidth=2, markersize=6)
    axes[0].plot(df_valid['seq_len'], df_valid['hy_time_ms'], 'r-s', label='Hyena O(L log L)', linewidth=2, markersize=6)
    # Thêm reference lines lý thuyết
    L_ref = df_valid['seq_len'].values
    ref_base = df_valid['tf_time_ms'].values[0]
    L0 = L_ref[0]
    axes[0].plot(L_ref, ref_base * (L_ref/L0)**2, 'b--', alpha=0.4, label='O(L²) theory')
    axes[0].plot(L_ref, ref_base * (L_ref/L0 * (L_ref/L0).round(2)), 'r--', alpha=0.4, label='O(L log L) theory')

axes[0].set_title('Forward Pass Time vs Sequence Length', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Sequence Length'); axes[0].set_ylabel('Time per step (ms)')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

# ── Plot 2: Memory vs Seq Length ─────────────────────────────
if len(df_valid) > 0:
    axes[1].plot(df_valid['seq_len'], df_valid['tf_mem_mb'], 'b-o', label='Transformer', linewidth=2, markersize=6)
    axes[1].plot(df_valid['seq_len'], df_valid['hy_mem_mb'], 'r-s', label='Hyena', linewidth=2, markersize=6)

axes[1].set_title('Peak GPU Memory vs Sequence Length', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Sequence Length'); axes[1].set_ylabel('Peak Memory (MB)')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

plt.suptitle('E2+E3: Scaling Comparison — Transformer vs Hyena', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results/plots/scaling_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: results/plots/scaling_comparison.png')

## Cell 12: Tổng Kết

### Kết quả mong đợi:
| Tiêu chí | Transformer | Hyena |
|---|---|---|
| Perplexity (WikiText-2) | Baseline | Tương đương ±10% |
| Time khi L tăng 2x | Tăng ~4x | Tăng ~2x |
| Memory khi L tăng 2x | Tăng ~4x | Tăng ~2x |

### Các file output:
- `results/E1_baseline.csv` — Loss/PPL theo epoch
- `results/scaling_comparison.csv` — Time/Memory vs L
- `results/plots/loss_curves.png` — Loss curves
- `results/plots/scaling_comparison.png` — Scaling charts

> **Lưu ý:** Kết quả có thể khác tùy theo số epoch, hyperparameter, và GPU. Mục tiêu là thấy **xu hướng** (trend), không phải số tuyệt đối.

In [ ]:
# List tất cả files kết quả đã tạo
import os
print('=== Output Files ===')
for root, dirs, files in os.walk('results'):
    for f in files:
        path = os.path.join(root, f)
        size = os.path.getsize(path)
        print(f'  {path:<50} ({size:,} bytes)')
print('\n✅ Notebook hoàn thành!')